`fmt_number` and `FmtOpts`
==============================================================================


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import splatlog as slog
from splatlog.rich import enrich, enrich_routine, highlighted

C = slog.rich.to_console(
    theme=slog.rich.THEME_ANSI_DARK,
    force_jupyter=True,
    color_system="truecolor",
)
p = C.print


def r(ren):
    return list(C.render(ren))


class Sample:
    """Fixture class exercising the three method kinds."""

    def instance_method(self):
        pass

    @classmethod
    def class_method(cls):
        pass

    @staticmethod
    def static_method():
        pass


def _make_nested():
    def inner():
        pass

    return inner

In [3]:
slog.rich.THEME_ANSI_DARK.styles["default"]

Style(color=Color('#dee1de', ColorType.TRUECOLOR, triplet=ColorTriplet(red=222, green=225, blue=222)))

In [4]:
from rich import box
from rich.pretty import Pretty
from rich.table import Table
from rich.text import Text


table = Table(box=box.HORIZONTALS, padding=(0, 1, 1, 1))

# Different "icon" text to try out
i_1 = "𝟊𝛘"
i_2 = "𝟊𝝌"
i_3 = "𝒇𝒙"

fn_icon = Text(f"{i_3} ", style="yellow")

for name, obj, opts in [
    ("Module function", slog.setup, {}),
    ("Function", slog.setup, {"fqn": False}),
    ("Unbound instance method", Sample.instance_method, {"fqn": True}),
    ("Bound instance method", Sample().instance_method, {"fqn": False}),
    ("Class method", Sample.class_method, {"fqn": False}),
]:
    text = enrich_routine(obj, fn_icon=fn_icon, **opts)
    assert isinstance(text, Text)
    table.add_row(
        name,
        text,
        Pretty(text.spans),
        # Pretty(r(text)),
    )

p(table)

──────────────────────────────────────────────────────────────────────────────────────────────────────── 
                                                                                                          
                                                                                                          
 ──────────────────────────────────────────────────────────────────────────────────────────────────────── 
  Module function           𝒇𝒙 splatlog.setup                    [                                        
                                                                     Span(0, 3, 'yellow'),                
                                                                     Span(3, 11, 'routine.module'),       
                                                                     Span(11, 12, 'routine.sep'),         
                                                                     Span(12, 17, 'routine.function')     
                                                                 ]                                        
                                                                                                          
  Function                  𝒇𝒙 setup                             [                                        
                                                                     Span(0, 3, 'yellow'),                
                                                                     Span(3, 8, 'routine.function')       
                                                                 ]                                        
                                                                                                          
  Unbound instance method   𝒇𝒙 __main__.Sample.instance_method   [                                        
                                                                     Span(0, 3, 'yellow'),                
                                                                     Span(3, 11, 'routine.module'),       
                                                                     Span(11, 12, 'routine.sep'),         
                                                                     Span(12, 18, 'routine.class'),       
                                                                     Span(18, 19, 'routine.sep'),         
                                                                     Span(19, 34, 'routine.method')       
                                                                 ]                                        
                                                                                                          
  Bound instance method     𝒇𝒙 Sample.instance_method            [                                        
                                                                     Span(0, 3, 'yellow'),                
                                                                     Span(3, 9, 'routine.class'),         
                                                                     Span(9, 10, 'routine.sep'),          
                                                                     Span(10, 25, 'routine.method')       
                                                                 ]                                        
                                                                                                          
  Class method              𝒇𝒙 Sample.class_method               [                                        
                                                                     Span(0, 3, 'yellow'),                
                                                                     Span(3, 9, 'routine.class'),         
                                                                     Span(9, 10, 'routine.sep'),          
                                                                     Span(10, 22, 'routine.classmethod')  
                                          

In [5]:
repr_s = """
[                                        
    Span(0, 3, 'routine.icon'),          
    Span(3, 11, 'routine.module'),       
    Span(11, 12, 'routine.sep'),         
    Span(12, 17, 'routine.function')     
] 
"""

text = highlighted(repr_s)
p(text)

[                                        
    Span(0, 3, 'routine.icon'),          
    Span(3, 11, 'routine.module'),       
    Span(11, 12, 'routine.sep'),         
    Span(12, 17, 'routine.function')     
]

In [6]:
def breakdown(text: Text):
    t = Table("Segment", "Style", "Resolved")
    for span in text.spans:
        t.add_row(
            Text(text.plain[span.start : span.end], style=span.style),
            Text(str(span.style)),
            slog.rich.repr_highlight(
                C.get_style(span.style)
                if isinstance(span.style, str)
                else None,
            ),
        )
    p(t)

In [7]:
breakdown(text)

┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Segment            ┃ Style       ┃ Resolved                                                                     ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ [                  │ repr.brace  │ Style(bold=True)                                                             │
│ (                  │ repr.brace  │ Style(bold=True)                                                             │
│ )                  │ repr.brace  │ Style(bold=True)                                                             │
│ (                  │ repr.brace  │ Style(bold=True)                                                             │
│ )                  │ repr.brace  │ Style(bold=True)                                                             │
│ (                  │ repr.brace  │ Style(bold=True)                                                             │
│ )                  │ repr.brace  │ Style(bold=True)                                                             │
│ (                  │ repr.brace  │ Style(bold=True)                                                             │
│ )                  │ repr.brace  │ Style(bold=True)                                                             │
│ ]                  │ repr.brace  │ Style(bold=True)                                                             │
│ Span               │ repr.call   │ Style(color=Color('#c678dd', ColorType.TRUECOLOR,                            │
│                    │             │ triplet=ColorTriplet(red=198, green=120, blue=221)), bold=True)              │
│ 0                  │ repr.number │ Style(color=Color('#56b6c2', ColorType.TRUECOLOR,                            │
│                    │             │ triplet=ColorTriplet(red=86, green=182, blue=194)), bold=True, italic=False) │
│ 3                  │ repr.number │ Style(color=Color('#56b6c2', ColorType.TRUECOLOR,                            │
│                    │             │ triplet=ColorTriplet(red=86, green=182, blue=194)), bold=True, italic=False) │
│ 'routine.icon'     │ repr.str    │ Style(color=Color('#98c379', ColorType.TRUECOLOR,                            │
│                    │             │ triplet=ColorTriplet(red=152, green=195, blue=121)), bold=False,             │
│                    │             │ italic=False)                                                                │
│ Span               │ repr.call   │ Style(color=Color('#c678dd', ColorType.TRUECOLOR,                            │
│                    │             │ triplet=ColorTriplet(red=198, green=120, blue=221)), bold=True)              │
│ 3                  │ repr.number │ Style(color=Color('#56b6c2', ColorType.TRUECOLOR,                            │
│                    │             │ triplet=ColorTriplet(red=86, green=182, blue=194)), bold=True, italic=False) │
│ 11                 │ repr.number │ Style(color=Color('#56b6c2', ColorType.TRUECOLOR,                            │
│                    │             │ triplet=ColorTriplet(red=86, green=182, blue=194)), bold=True, italic=False) │
│ 'routine.module'   │ repr.str    │ Style(color=Color('#98c379', ColorType.TRUECOLOR,                            │
│                    │             │ triplet=ColorTriplet(red=152, green=195, blue=121)), bold=False,             │
│                    │             │ italic=False)                                                                │
│ Span               │ repr.call   │ Style(color=Color('#c678dd', ColorType.TRUECOLOR,                            │
│                    │             │ triplet=ColorTriplet(red=198, green=120, blue=221)), bold=True)              │
│ 11                 │ repr.number │ Style(color=Color('#56b6c2', ColorType.TRUECOLOR,                            │
│                    │             │ triplet=ColorTriple

In [8]:
class A:
    pass


breakdown(highlighted(repr(A())))


┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Segment                ┃ Style             ┃ Resolved                                                           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ <                      │ repr.tag_start    │ Style(bold=True)                                                   │
│ __main__.A             │ repr.tag_name     │ Style(color=Color('#b95bde', ColorType.TRUECOLOR,                  │
│                        │                   │ triplet=ColorTriplet(red=185, green=91, blue=222)), bold=True)     │
│  object at 0x10a3f6ba0 │ repr.tag_contents │ Style(color=Color('default', ColorType.DEFAULT))                   │
│ >                      │ repr.tag_end      │ Style(bold=True)                                                   │
│ 0x10a3f6ba0            │ repr.number       │ Style(color=Color('#56b6c2', ColorType.TRUECOLOR,                  │
│                        │                   │ triplet=ColorTriplet(red=86, green=182, blue=194)), bold=True,     │
│                        │                   │ italic=False)                                                      │
└────────────────────────┴───────────────────┴────────────────────────────────────────────────────────────────────┘